# Experiências de janelas e agrupamentos — BERTilda

Reutiliza os embeddings contextual e lexical de 2015–2017. Compara janelas anuais e semestrais com três configurações de agrupamento; os parâmetros diferentes são testados separadamente. As relações temporais são uma aproximação exploratória ao BERTilda. Esta experiência pode demorar bastante: ajusta `JANELAS`, `CONFIGURACOES` ou `VERSOES` antes de executar.


In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from bertopic import BERTopic
from hdbscan import HDBSCAN
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import adjusted_rand_score
from umap import UMAP
from IPython.display import display


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RANDOM_STATE = 42
YEARS = [2015, 2016, 2017]
VERSOES = ["contextual", "lexical"]
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MAX_SEQ_LENGTH = 512
MIN_CLUSTER_SIZE = 50
MIN_DOCS_PER_TOPIC = 50
TOP_N_WORDS = 10
SIMILARITY_THRESHOLD = 0.70
DOCUMENT_TOPIC_THRESHOLD = 0.40
COVERAGE_THRESHOLD = 0.30
SPLIT_COVERAGE_THRESHOLD = 0.70
MERGE_COVERAGE_THRESHOLD = 0.70
VALIDATION_SAMPLE_SIZE = 30
NOME_EXPERIENCIA = "experiencias_janelas_parametros_v1"

# A configuração de referência é comparada com uma alteração de cada vez.
CONFIGURACOES = {
    "referencia": {"vizinhos": 15, "selecao": "eom"},
    "umap_local": {"vizinhos": 8, "selecao": "eom"},
    "hdbscan_leaf": {"vizinhos": 15, "selecao": "leaf"},
}
JANELAS = ["anual", "semestral"]


## Carregar e verificar embeddings

Os IDs, os textos e os blocos são verificados. O MPS não é necessário: não são criados embeddings.


In [3]:
data_dir = next((p.resolve() for p in [Path("../../data"), Path("data")]
                 if (p / "parlamint_pt_corpus_contextual.csv").exists()), None)
if data_dir is None:
    raise FileNotFoundError("Não foram encontrados os corpus em data.")
nome_anos = "anos_" + "_".join(map(str, YEARS))
embedding_base = data_dir / "embeddings_bertimbau" / nome_anos
output_base = data_dir / "bertilda_annual" / NOME_EXPERIENCIA
if len(set(YEARS)) != len(YEARS) or YEARS != sorted(YEARS) or any(b - a != 1 for a, b in zip(YEARS, YEARS[1:])):
    raise ValueError("YEARS deve conter anos únicos, consecutivos e por ordem crescente.")

def assinatura_corpus(ids, textos, versao, chunk_size):
    h = hashlib.sha256()
    config = {"modelo": MODEL_NAME, "max_seq_length": MAX_SEQ_LENGTH, "pooling": "mean",
              "normalize_embeddings": True, "versao": versao, "chunk_size": chunk_size}
    h.update(json.dumps(config, sort_keys=True).encode())
    for identificador, texto in zip(ids, textos):
        for valor in (identificador, texto):
            dados = valor.encode("utf-8")
            h.update(len(dados).to_bytes(8, "big"))
            h.update(dados)
    return h.hexdigest()

def carregar_versao(versao):
    ficheiro = data_dir / f"parlamint_pt_corpus_{versao}.csv"
    corpus = pd.read_csv(ficheiro, dtype="string", keep_default_na=False)
    if not {"ID", "Date", "texto_modelo"}.issubset(corpus.columns):
        raise ValueError(f"Colunas necessárias em falta: {ficheiro}")
    corpus["ano"] = pd.to_datetime(corpus["Date"], errors="coerce").dt.year
    corpus = corpus.loc[corpus["ano"].isin(YEARS)].sort_values("ID").reset_index(drop=True)
    if corpus.empty or corpus["ID"].eq("").any() or corpus["ID"].duplicated().any() or corpus["texto_modelo"].str.strip().eq("").any():
        raise ValueError(f"Corpus vazio ou com IDs/textos inválidos: {ficheiro}")
    if set(corpus["ano"].unique()) != set(YEARS):
        raise ValueError(f"Faltam anos em {ficheiro}")
    pasta = embedding_base / versao
    manifesto = json.loads((pasta / "manifesto.json").read_text(encoding="utf-8"))
    chunk_size = int(manifesto["chunk_size"])
    ids = corpus["ID"].astype(str).tolist()
    textos = corpus["texto_modelo"].astype(str).tolist()
    assinatura = assinatura_corpus(ids, textos, versao, chunk_size)
    if manifesto["assinatura"] != assinatura or manifesto["documentos"] != len(ids) or manifesto["modelo"] != MODEL_NAME or manifesto["versao"] != versao:
        raise ValueError(f"O manifesto não corresponde ao corpus ou modelo: {pasta}")
    blocos = []
    dimensao = None
    for inicio in range(0, len(ids), chunk_size):
        fim = min(inicio + chunk_size, len(ids))
        caminho = pasta / f"bloco_{inicio:08d}_{fim:08d}.npz"
        if not caminho.exists():
            raise FileNotFoundError(f"Bloco de embeddings em falta: {caminho}")
        with np.load(caminho, allow_pickle=False) as bloco:
            vetores = bloco["embeddings"]
            if not np.array_equal(bloco["document_ids"].astype(str), np.asarray(ids[inicio:fim])):
                raise ValueError(f"IDs desalinhados no bloco: {caminho}")
            if vetores.ndim != 2 or vetores.shape[0] != fim - inicio or not np.isfinite(vetores).all():
                raise ValueError(f"Embeddings inválidos: {caminho}")
            if dimensao is None:
                dimensao = vetores.shape[1]
            if vetores.shape[1] != dimensao:
                raise ValueError(f"Dimensão inconsistente: {caminho}")
            blocos.append(vetores)
    corpus = corpus.rename(columns={"texto_modelo": "documento_modelo"})
    return corpus, np.concatenate(blocos, axis=0), assinatura

corpora = {}
for versao in VERSOES:
    corpora[versao] = carregar_versao(versao)
    print(f"{versao}: {len(corpora[versao][0]):,} intervenções; embeddings {corpora[versao][1].shape}")
if not corpora["contextual"][0]["ID"].equals(corpora["lexical"][0]["ID"]):
    raise ValueError("Os corpus não têm os mesmos IDs na mesma ordem.")
if not corpora["contextual"][0]["ano"].equals(corpora["lexical"][0]["ano"]):
    raise ValueError("As datas dos mesmos IDs divergem entre corpus.")
print("IDs, anos, textos e blocos verificados.")


contextual: 36,258 intervenções; embeddings (36258, 768)
lexical: 36,258 intervenções; embeddings (36258, 768)
IDs, anos, textos e blocos verificados.


In [4]:
try:
    portuguese_stopwords = stopwords.words("portuguese")
except LookupError as error:
    raise LookupError("Instale os stopwords com nltk.download('stopwords') e volte a executar esta célula.") from error

def make_topic_model():
    umap_model = UMAP(
        n_neighbors=UMAP_NEIGHBORS,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=RANDOM_STATE,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        metric="euclidean",
        prediction_data=True,
        cluster_selection_method=HDBSCAN_SELECTION,
    )
    vectorizer_model = CountVectorizer(
        stop_words=portuguese_stopwords,
        ngram_range=(1, 2),
        min_df=2,
    )
    return BERTopic(
        language="multilingual",
        embedding_model=None,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False,
        verbose=True,
    )

In [5]:
def forward_coverage(window_1, window_2):
    coverage = {}
    target_ids = list(window_2["topic_embeddings"].keys())
    if not target_ids:
        return coverage

    target_centroids = np.vstack([window_2["topic_embeddings"][topic] for topic in target_ids])
    source_docs = window_1["documents_with_topics"]

    for source_topic in window_1["topic_embeddings"]:
        embeddings = np.vstack(source_docs.loc[source_docs["topic_id"] == source_topic, "embedding"].to_numpy())
        similarity = cosine_similarity(embeddings, target_centroids)
        best_target = similarity.argmax(axis=1)
        best_score = similarity.max(axis=1)
        valid = best_score >= DOCUMENT_TOPIC_THRESHOLD
        counts = np.bincount(best_target[valid], minlength=len(target_ids))
        for index, count in enumerate(counts):
            if count:
                coverage[(source_topic, target_ids[index])] = count / len(embeddings)
    return coverage

def backward_coverage(window_1, window_2):
    coverage = {}
    source_ids = list(window_1["topic_embeddings"].keys())
    if not source_ids:
        return coverage

    source_centroids = np.vstack([window_1["topic_embeddings"][topic] for topic in source_ids])
    target_docs = window_2["documents_with_topics"]

    for target_topic in window_2["topic_embeddings"]:
        embeddings = np.vstack(target_docs.loc[target_docs["topic_id"] == target_topic, "embedding"].to_numpy())
        similarity = cosine_similarity(embeddings, source_centroids)
        best_source = similarity.argmax(axis=1)
        best_score = similarity.max(axis=1)
        valid = best_score >= DOCUMENT_TOPIC_THRESHOLD
        counts = np.bincount(best_source[valid], minlength=len(source_ids))
        for index, count in enumerate(counts):
            if count:
                coverage[(source_ids[index], target_topic)] = count / len(embeddings)
    return coverage

## Experiências e amostras para leitura

Para cada cenário são guardados diagnósticos, descritores, atribuições, relações e até três textos por tópico. As janelas semestrais usam limites de janeiro–junho e julho–dezembro; relações ligam janelas adjacentes. Os resultados existentes são preservados, mas a execução interrompida exige novo nome de experiência.


In [6]:
def executar_versao(versao, janela_tipo, nome_config, configuracao):
    print(f"\n=== {versao.upper()} ===")
    corpus_run, document_embeddings, assinatura = corpora[versao]
    global UMAP_NEIGHBORS, HDBSCAN_SELECTION
    UMAP_NEIGHBORS = configuracao["vizinhos"]
    HDBSCAN_SELECTION = configuracao["selecao"]
    datas = pd.to_datetime(corpus_run["Date"], errors="raise")
    if janela_tipo == "anual":
        periodos = datas.dt.year.astype(str)
    elif janela_tipo == "semestral":
        periodos = datas.dt.year.astype(str) + "-S" + ((datas.dt.month - 1) // 6 + 1).astype(str)
    else:
        raise ValueError(f"Janela desconhecida: {janela_tipo}")
    janelas = sorted(periodos.unique())
    esperado = [str(ano) for ano in YEARS] if janela_tipo == "anual" else [f"{ano}-S{s}" for ano in YEARS for s in (1, 2)]
    if janelas != esperado:
        raise ValueError(f"Faltam janelas: {sorted(set(esperado) - set(janelas))}")
    corpus_run = corpus_run.copy()
    corpus_run["janela"] = periodos
    window_topics = []
    assignments = []

    for year in janelas:
        indices = np.flatnonzero(corpus_run["janela"].to_numpy() == year)
        subset = corpus_run.iloc[indices].copy().reset_index(drop=True)
        docs_window = subset["documento_modelo"].tolist()
        embeddings_window = document_embeddings[indices]

        topic_model = make_topic_model()
        topics, _ = topic_model.fit_transform(docs_window, embeddings=embeddings_window)

        subset["topic_id"] = topics
        subset["embedding"] = list(embeddings_window)
        assignments.append(subset)

        topic_info = topic_model.get_topic_info()
        valid_topic_ids = topic_info.loc[
            (topic_info["Topic"] != -1) & (topic_info["Count"] >= MIN_DOCS_PER_TOPIC),
            "Topic",
        ].tolist()

        topic_embeddings = {
            topic_id: embeddings_window[np.array(topics) == topic_id].mean(axis=0)
            for topic_id in valid_topic_ids
        }
        topic_words = {
            topic_id: [word for word, _ in topic_model.get_topic(topic_id)[:TOP_N_WORDS]]
            for topic_id in valid_topic_ids
        }
        topic_doc_counts = {
            int(row.Topic): int(row.Count)
            for row in topic_info.itertuples()
            if row.Topic in valid_topic_ids
        }

        window_topics.append({
            "year": year,
            "num_docs": len(subset),
            "num_noise_docs": int((subset["topic_id"] == -1).sum()),
            "topic_words": topic_words,
            "topic_doc_counts": topic_doc_counts,
            "topic_embeddings": topic_embeddings,
            "documents_with_topics": subset,
        })

    topic_assignments = pd.concat(assignments, ignore_index=True)
    print(f"Janelas modeladas: {len(window_topics)}")

    diagnostics = pd.DataFrame([
        {
            "janela": window["year"],
            "documentos": window["num_docs"],
            "topicos_validos": len(window["topic_embeddings"]),
            "documentos_ruido": window["num_noise_docs"],
            "proporcao_ruido": window["num_noise_docs"] / window["num_docs"],
        }
        for window in window_topics
    ]).sort_values("janela")

    display(diagnostics)

    if (diagnostics["topicos_validos"] == 0).any():
        print("Aviso: há anos sem tópicos válidos. Reveja MIN_CLUSTER_SIZE e MIN_DOCS_PER_TOPIC.")
    if (diagnostics["proporcao_ruido"] > 0.50).any():
        print("Aviso: há anos com mais de 50% de documentos classificados como ruído.")

    edges = []
    events = []

    for window_1, window_2 in zip(window_topics[:-1], window_topics[1:]):
        source_ids = list(window_1["topic_embeddings"].keys())
        target_ids = list(window_2["topic_embeddings"].keys())
        if not source_ids or not target_ids:
            continue

        source_centroids = np.vstack([window_1["topic_embeddings"][topic] for topic in source_ids])
        target_centroids = np.vstack([window_2["topic_embeddings"][topic] for topic in target_ids])
        similarities = cosine_similarity(source_centroids, target_centroids)
        coverage_forward = forward_coverage(window_1, window_2)
        coverage_backward = backward_coverage(window_1, window_2)

        for source_index, source_topic in enumerate(source_ids):
            for target_index, target_topic in enumerate(target_ids):
                similarity = similarities[source_index, target_index]
                forward = coverage_forward.get((source_topic, target_topic), 0)
                backward = coverage_backward.get((source_topic, target_topic), 0)
                if (
                    similarity >= SIMILARITY_THRESHOLD
                    and forward >= COVERAGE_THRESHOLD
                    and backward >= COVERAGE_THRESHOLD
                ):
                    edges.append({
                        "janela_origem": window_1["year"],
                        "topico_origem": source_topic,
                        "palavras_origem": ", ".join(window_1["topic_words"][source_topic]),
                        "janela_destino": window_2["year"],
                        "topico_destino": target_topic,
                        "palavras_destino": ", ".join(window_2["topic_words"][target_topic]),
                        "similaridade": similarity,
                        "cobertura_forward": forward,
                        "cobertura_backward": backward,
                    })

    edges_df = pd.DataFrame(edges)

    for window_1, window_2 in zip(window_topics[:-1], window_topics[1:]):
        links = edges_df[
            (edges_df["janela_origem"] == window_1["year"])
            & (edges_df["janela_destino"] == window_2["year"])
        ] if not edges_df.empty else pd.DataFrame()

        for source_topic in window_1["topic_embeddings"]:
            successors = links[links["topico_origem"] == source_topic] if not links.empty else links
            if successors.empty:
                relation = "desaparecimento"
            elif len(successors) == 1:
                relation = "continuação"
            elif successors["cobertura_forward"].sum() >= SPLIT_COVERAGE_THRESHOLD:
                relation = "divisão"
            else:
                relation = "ambíguo"

            events.append({
                "tipo": relation,
                "janela_origem": window_1["year"],
                "topico_origem": source_topic,
                "palavras_origem": ", ".join(window_1["topic_words"][source_topic]),
                "janela_destino": window_2["year"],
                "topicos_destino": successors["topico_destino"].tolist() if not successors.empty else [],
                "palavras_destino": successors["palavras_destino"].tolist() if not successors.empty else [],
                "cobertura_forward_total": successors["cobertura_forward"].sum() if not successors.empty else 0,
                "cobertura_backward_total": successors["cobertura_backward"].sum() if not successors.empty else 0,
            })

        for target_topic in window_2["topic_embeddings"]:
            predecessors = links[links["topico_destino"] == target_topic] if not links.empty else links
            if (
                len(predecessors) >= 2
                and predecessors["cobertura_backward"].sum() >= MERGE_COVERAGE_THRESHOLD
            ):
                events.append({
                    "tipo": "fusão",
                    "janela_origem": window_1["year"],
                    "topico_origem": predecessors["topico_origem"].tolist(),
                    "palavras_origem": predecessors["palavras_origem"].tolist(),
                    "janela_destino": window_2["year"],
                    "topicos_destino": [target_topic],
                    "palavras_destino": [", ".join(window_2["topic_words"][target_topic])],
                    "cobertura_forward_total": predecessors["cobertura_forward"].sum(),
                    "cobertura_backward_total": predecessors["cobertura_backward"].sum(),
                })

    events_df = pd.DataFrame(events)
    if not events_df.empty:
        display(events_df["tipo"].value_counts().rename("eventos").to_frame())

    if events_df.empty:
        validation_sample = events_df.copy()
    else:
        validation_sample = events_df.sample(
            n=min(VALIDATION_SAMPLE_SIZE, len(events_df)),
            random_state=RANDOM_STATE,
        ).sort_values(["janela_origem", "tipo"])

    display(validation_sample)

    topicos = pd.DataFrame([
        {"janela": w["year"], "topico": t, "intervencoes": w["topic_doc_counts"][t],
         "palavras": ", ".join(w["topic_words"][t])}
        for w in window_topics for t in w["topic_words"]
    ], columns=["janela", "topico", "intervencoes", "palavras"])
    output_dir = output_base / janela_tipo / nome_config / versao
    output_dir.mkdir(parents=True, exist_ok=False)
    diagnostics.to_csv(output_dir / "diagnosticos.csv", index=False)
    topic_assignments.drop(columns="embedding").to_csv(output_dir / "atribuicoes.csv", index=False)
    topicos.to_csv(output_dir / "topicos.csv", index=False)
    edges_df.to_csv(output_dir / "ligacoes.csv", index=False)
    events_df.to_csv(output_dir / "eventos.csv", index=False)
    validation_sample.to_csv(output_dir / "amostra_relacoes.csv", index=False)
    # Amostra aleatória de até três intervenções por tópico válido, incluindo texto integral.
    validos = topicos[["janela", "topico"]].rename(columns={"topico": "topic_id"})
    atribuicoes = topic_assignments.drop(columns="embedding")
    candidatos = atribuicoes.merge(validos, on=["janela", "topic_id"], validate="many_to_one")
    grupos_amostras = [grupo.sample(n=min(3, len(grupo)), random_state=RANDOM_STATE)
                       for _, grupo in candidatos.groupby(["janela", "topic_id"])]
    amostras = pd.concat(grupos_amostras, ignore_index=True) if grupos_amostras else candidatos.copy()
    amostras.to_csv(output_dir / "amostra_intervencoes.csv", index=False)
    tamanhos = topicos.groupby("janela")["intervencoes"].max().rename("maior_topico")
    diagnosticos_extra = diagnostics.merge(tamanhos, on="janela", how="left")
    diagnosticos_extra["proporcao_maior_topico"] = diagnosticos_extra["maior_topico"] / diagnosticos_extra["documentos"]
    diagnosticos_extra.to_csv(output_dir / "diagnosticos_completos.csv", index=False)
    (output_dir / "configuracao.json").write_text(json.dumps({"janela": janela_tipo, "configuracao": nome_config,
        "vizinhos": UMAP_NEIGHBORS, "selecao_hdbscan": HDBSCAN_SELECTION,
        "min_cluster_size": MIN_CLUSTER_SIZE, "anos": YEARS, "assinatura": assinatura,
        "language": "multilingual", "semente": RANDOM_STATE}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Resultados guardados: {output_dir}")
    return diagnostics, topic_assignments.drop(columns="embedding"), topicos, events_df, assinatura


In [7]:
resultados = []
for janela_tipo in JANELAS:
    for nome_config, configuracao in CONFIGURACOES.items():
        for versao in VERSOES:
            destino = output_base / janela_tipo / nome_config / versao
            if destino.exists():
                raise FileExistsError(f"Resultados existentes preservados: {destino}. Escolhe outro NOME_EXPERIENCIA.")
            print(f"\n{janela_tipo} | {nome_config} | {versao}")
            diagnosticos, atribuicoes, topicos, eventos, assinatura = executar_versao(
                versao, janela_tipo, nome_config, configuracao)
            tabela = pd.read_csv(destino / "diagnosticos_completos.csv")
            tabela["janela_tipo"] = janela_tipo
            tabela["configuracao"] = nome_config
            tabela["versao"] = versao
            resultados.append(tabela)
comparacao = pd.concat(resultados, ignore_index=True)
comparacao.to_csv(output_base / "comparacao_diagnosticos.csv", index=False)
display(comparacao.sort_values(["versao", "janela_tipo", "configuracao", "janela"]))


2026-10-25 20:21:42,169 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm



anual | referencia | contextual

=== CONTEXTUAL ===


2026-10-25 20:21:56,495 - BERTopic - Dimensionality - Completed ✓
2026-10-25 20:21:56,496 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-25 20:21:56,875 - BERTopic - Cluster - Completed ✓
2026-10-25 20:21:56,878 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-25 20:21:57,911 - BERTopic - Representation - Completed ✓
2026-10-25 20:21:58,060 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-25 20:22:05,320 - BERTopic - Dimensionality - Completed ✓
2026-10-25 20:22:05,322 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-25 20:22:06,530 - BERTopic - Cluster - Completed ✓
2026-10-25 20:22:06,537 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-25 20:22:08,722 - BERTopic - Representation - Completed ✓
2026-10-25 20:22:08,970 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-25 20:22:19,469 - BERTopic - D

Janelas modeladas: 3


,janela,documentos,topicos_validos,documentos_ruido,proporcao_ruido
0,2015,11379,4,150,0.013182
1,2016,12104,6,164,0.013549
2,2017,12775,7,160,0.012524


,eventos
tipo,
continuação,8
divisão,1
desaparecimento,1


... (81 outputs left out)

,eventos
tipo,
continuação,49
desaparecimento,22
divisão,2


,tipo,janela_origem,topico_origem,palavras_origem,janela_destino,topicos_destino,palavras_destino,cobertura_forward_total,cobertura_backward_total
4,continuação,2015-S1,4,"declaração voto, voto, declaração, apresentar ...",2015-S2,[4],"[declaração voto, declaração, voto, votação, a...",0.890909,0.957983
0,desaparecimento,2015-S1,0,"saúde, serviço, urgência, hospital, nacional s...",2015-S2,[],[],0.000000,0.000000
12,desaparecimento,2015-S1,12,"república, assembleia, liberdade, assembleia r...",2015-S2,[],[],0.000000,0.000000
9,desaparecimento,2015-S1,9,"xii, baixar, comissão, baixar comissão, pcp, r...",2015-S2,[],[],0.000000,0.000000
5,desaparecimento,2015-S1,5,"social, trabalhador, pobreza, governo, trabalh...",2015-S2,[],[],0.000000,0.000000
7,desaparecimento,2015-S1,7,"partido, socialista, partido socialista, portu...",2015-S2,[],[],0.000000,0.000000
10,divisão,2015-S1,10,"grécia, governo, grego, país, europa, europeia...",2015-S2,"[0, 1]","[governo, social, ano, país, português, públic...",1.000000,0.948852
18,continuação,2015-S2,2,"antónio, partido, querer, governo, antónio cos...",2016-S1,[7],"[psd, cds, joão, governo, psd cds, paulo, perg...",0.641414,0.958763
22,continuação,2015-S2,6,"peçar desculpa, peçar, desculpa, pcp, interpel...",2016-S1,[1],"[governo, português, debate, facto, vou, país,...",0.956522,0.713068
16,continuação,2015-S2,0,"governo, social, ano, país, português, público...",2016-S1,[9],"[empresa, orçamento, rendimento, aumento, famí...",0.397541,0.978723


Resultados guardados: /Users/alexandresousa/Desktop/Tese/data/bertilda_annual/experiencias_janelas_parametros_v1/semestral/hdbscan_leaf/lexical


,janela,documentos,topicos_validos,documentos_ruido,proporcao_ruido,maior_topico,proporcao_maior_topico,janela_tipo,configuracao,versao
12,2015,11379,15,8853,0.778012,453,0.039810,anual,hdbscan_leaf,contextual
13,2016,12104,21,9398,0.776438,515,0.042548,anual,hdbscan_leaf,contextual
14,2017,12775,21,9726,0.761331,388,0.030372,anual,hdbscan_leaf,contextual
0,2015,11379,4,150,0.013182,10753,0.944986,anual,referencia,contextual
1,2016,12104,6,164,0.013549,11394,0.941342,anual,referencia,contextual
2,2017,12775,7,160,0.012524,11944,0.934951,anual,referencia,contextual
6,2015,11379,4,147,0.012919,10745,0.944283,anual,umap_local,contextual
7,2016,12104,4,39,0.003222,11381,0.940268,anual,umap_local,contextual
8,2017,12775,5,170,0.013307,12169,0.952564,anual,umap_local,contextual
42,2015-S1,8216,12,5671,0.690239,937,0.114046,semestral,hdbscan_leaf,contextual


## Como interpretar

Compara número de intervenções, tópicos, ruído e proporção do maior tópico por janela. Lê `amostra_intervencoes.csv` para avaliar coerência. Compara as relações em `ligacoes.csv`, `eventos.csv` e `amostra_relacoes.csv`, tendo em conta que janelas semestrais geram mais transições e que as regras são aproximações. Mais tópicos ou menos ruído não provam melhor qualidade. Um resultado isolado com a mesma semente não mede estabilidade; repete cenários promissores com outras sementes.
